In [ ]:
import numpy as np
from collections import defaultdict

# Sample dataset for text classification
# Each entry is (text, label)
corpus = [
    ("Chinese Beijing Chinese", "yes"),
    ("Chinese Chinese Shanghai", "yes"),
    ("Chinese Macao", "yes"),
    ("Tokyo Japan Chinese", "no")
]

# Preprocessing: Tokenize and build vocabulary
def tokenize(text):
    return text.lower().split()

def build_vocabulary(corpus):
    vocab = set()
    for text, _ in corpus:
        vocab.update(tokenize(text))
    return list(vocab)

vocabulary = build_vocabulary(corpus)
print(f"Vocabulary: {vocabulary}")
print(f"Vocabulary size: {len(vocabulary)}")

Vocabulary: ['beijing', 'shanghai', 'macao', 'japan', 'tokyo', 'chinese']
Vocabulary size: 6


In [ ]:
class NaiveBayesClassifier:
    def __init__(self, alpha=1):
        self.alpha = 1  # Laplace smoothing parameter
        self.priors = {}
        self.likelihoods = defaultdict(lambda: defaultdict(float))
        self.vocab = []
        self.vocab_size = 0
        self.class_word_counts = defaultdict(lambda: defaultdict(int)) # Count of word in class
        self.class_total_words = defaultdict(int) # Total words in class

    def fit(self, X, y):
        # Ensure X is a list of tokenized documents and y is a list of labels
        if not X or not y or len(X) != len(y):
            raise ValueError("Input X and y must be non-empty and have the same length.")

        # Build vocabulary from training data
        all_words = set()
        for doc in X:
            all_words.update(doc)
        self.vocab = list(all_words)
        self.vocab_size = len(self.vocab)

        # Calculate priors P(C)
        total_documents = len(y)
        class_counts = defaultdict(int)
        for label in y:
            class_counts[label] += 1
        for label, count in class_counts.items():
            self.priors[label] = count / total_documents

        # Calculate word counts for likelihoods P(word|C)
        for doc, label in zip(X, y):
            for word in doc:
                self.class_word_counts[label][word] += 1
            self.class_total_words[label] += len(doc)

        # Calculate likelihoods P(word|C) with Laplace smoothing
        for label in self.priors.keys():
            for word in self.vocab:
                # (Count(word, C) + alpha) / (Count(C) + alpha * |V|)
                numerator = self.class_word_counts[label][word] + self.alpha
                denominator = self.class_total_words[label] + self.alpha * self.vocab_size
                self.likelihoods[label][word] = numerator / denominator

    def predict(self, X_test):
        predictions = []
        for doc in X_test:
            scores = {}
            for label, prior in self.priors.items():
                # Start with log(prior) to avoid underflow with many small probabilities
                log_score = np.log(prior)
                for word in doc:
                    # If word is unseen in training vocab, assign a very small probability
                    # This handles new words not in the training vocabulary, but different from zero-count words in training classes.
                    if word not in self.vocab:
                        # We need a fallback for truly unseen words (not in self.vocab at all)
                        # A common approach is to use 1/(total_words_in_class + alpha * |V|)
                        # Or, for simplicity and robustness, use the smoothed probability for an unseen word:
                        # alpha / (class_total_words[label] + alpha * vocab_size)
                        # However, for words *in* self.vocab but not *in* class, self.likelihoods already has the smoothed value
                        # For words *not* in self.vocab, we can use a very small constant or a default smoothed value.
                        # Let's use the smoothed probability for an unseen word: alpha / (class_total_words[label] + alpha * vocab_size)
                        unseen_word_prob = self.alpha / (self.class_total_words[label] + self.alpha * self.vocab_size)
                        log_score += np.log(unseen_word_prob)
                    else:
                        log_score += np.log(self.likelihoods[label][word])
                scores[label] = log_score
            predictions.append(max(scores, key=scores.get))
        return predictions

# --- Demonstration ---

# Prepare data for the classifier
X_train = [tokenize(text) for text, _ in corpus]
y_train = [label for _, label in corpus]

# Initialize and train the classifier
nb_classifier = NaiveBayesClassifier(alpha=1)
nb_classifier.fit(X_train, y_train)

print("\n--- Model Training Summary ---")
print("Priors (P(C)):", nb_classifier.priors)

for label in nb_classifier.priors.keys():
    print(f"\nLikelihoods for class '{label}' (P(word|{label})):")
    sorted_likelihoods = sorted(nb_classifier.likelihoods[label].items(), key=lambda item: item[0])
    for word, prob in sorted_likelihoods:
        print(f"  P('{word}'|'{label}') = {prob:.4f}")


# Test cases
X_test = [
    tokenize("Chinese Chinese Chinese Tokyo Japan"),
    tokenize("Chinese Chinese Chinese Chinese"), # Should be 'yes'
    tokenize("Japan Japan Japan"), # Should be 'no'
    tokenize("new word here"), # Test with entirely new words
    tokenize("Shanghai Chinese")
]

predictions = nb_classifier.predict(X_test)

print("\n--- Predictions ---")
for i, doc in enumerate(X_test):
    print(f"Document: {' '.join(doc)} -> Predicted: {predictions[i]}")


--- Model Training Summary ---
Priors (P(C)): {'yes': 0.75, 'no': 0.25}

Likelihoods for class 'yes' (P(word|yes)):
  P('beijing'|'yes') = 0.1429
  P('chinese'|'yes') = 0.4286
  P('japan'|'yes') = 0.0714
  P('macao'|'yes') = 0.1429
  P('shanghai'|'yes') = 0.1429
  P('tokyo'|'yes') = 0.0714

Likelihoods for class 'no' (P(word|no)):
  P('beijing'|'no') = 0.1111
  P('chinese'|'no') = 0.2222
  P('japan'|'no') = 0.2222
  P('macao'|'no') = 0.1111
  P('shanghai'|'no') = 0.1111
  P('tokyo'|'no') = 0.2222

--- Predictions ---
Document: chinese chinese chinese tokyo japan -> Predicted: yes
Document: chinese chinese chinese chinese -> Predicted: yes
Document: japan japan japan -> Predicted: no
Document: new word here -> Predicted: no
Document: shanghai chinese -> Predicted: yes
